# Install

In [1]:
!pip install -q \
langchain \
langchain-community \
langchain-google-genai \
langchain-text-splitters \
langchain-classic \
pageindex \
rank-bm25 \
reportlab

# Imports

In [2]:
import os
import json
import re
import time
import numpy as np

from google.colab import userdata

from pageindex import PageIndexClient

from rank_bm25 import BM25Okapi

from reportlab.platypus import SimpleDocTemplate, Paragraph, PageBreak

from reportlab.lib.styles import getSampleStyleSheet

from pydantic import Field

from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever

from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_core.prompts import ChatPromptTemplate

from langchain_classic.chains.combine_documents import create_stuff_documents_chain

from langchain_classic.chains import create_retrieval_chain

# API Setup

In [3]:
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
pi_client = PageIndexClient(api_key=userdata.get("PAGEINDEX"))
print("ok")

ok


# Load Corpus

In [4]:
with open("/content/corpus.json", "r", encoding="utf-8") as f:
    corpus = json.load(f)

print(f"Articles: {len(corpus)}")

Articles: 17


# Cleaning Corpus

In [ ]:
# This function cleans article text for preprocessing
# It lowercases, removes punctuation, and normalizes spaces
def clean_text(text):
    # Convert to lowercase for consistency
    text = text.lower()
    # Remove special characters and punctuation, keep alphanumeric and spaces
    text = re.sub(r"[^\w\s]", " ", text)
    # Replace multiple spaces with single space
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [6]:
cleaned_corpus = []

for article in corpus:
    cleaned_corpus.append(
        {
            "title": article["title"],
            "url": article["url"],
            "text": clean_text(article["content"])
        }
    )

print(f"Processed: {len(cleaned_corpus)}")

Processed: 17


# Make PDF and upload to PageIndex

In [7]:
pdf = SimpleDocTemplate("corpus.pdf")

styles = getSampleStyleSheet()

elements = []

for article in cleaned_corpus:

    elements.append(
        Paragraph(
            f"<b>{article['title']}</b>",
            styles["Heading2"]
        )
    )

    elements.append(
        Paragraph(
            article["text"],
            styles["BodyText"]
        )
    )

    elements.append(
        Paragraph(
            article["url"],
            styles["BodyText"]
        )
    )

    elements.append(
        PageBreak()
    )

pdf.build(elements)

print("PDF Created")

PDF Created


In [8]:
response = pi_client.submit_document("corpus.pdf")
doc_id = response["doc_id"]
print(doc_id)

pi-cmpwuaatt02vb01quqclhdgdh


# Wait for indexing

In [10]:
while True:

    info = pi_client.get_document(doc_id)
    status = info["status"]
    print(status)

    if status == "completed":
        break

    time.sleep(10)

processing
processing
processing
processing
completed


# Fetch Tree and Extract Nodes

In [11]:
tree = pi_client.get_tree(doc_id)
print(tree.keys())

dict_keys(['doc_id', 'status', 'retrieval_ready', 'result', 'metadata'])


In [12]:
documents = []

for node in tree["result"]:
    documents.append(
        {
            "node_id": node.get("node_id"),
            "title": node.get("title", ""),
            "text": node.get("text", ""),
            "parent": node.get("parent_id"),
            "level": node.get("level")
        }
    )

print(f"Nodes: {len(documents)}")

Nodes: 17


# BM25 Corpus

In [13]:
tokenized_corpus = [doc["text"].split() for doc in documents]

In [14]:
bm25 = BM25Okapi(tokenized_corpus)

# Retrieval

In [ ]:
# Retrieve top-k documents using BM25 ranking
# BM25 scores documents based on term frequency and inverse document frequency
def retrieve(query, k=5):
    # Tokenize query the same way as corpus
    query_tokens = (query.lower().split())
    # Calculate BM25 scores for all documents
    scores = bm25.get_scores(query_tokens)
    # Get indices of top-k documents with highest scores
    top_idx = np.argsort(scores)[::-1][:k]

    results = []
    # Return top results with scores
    for idx in top_idx:
        results.append(
            {
                "score":float(scores[idx]),
                "document":documents[idx]
            }
        )

    return results

In [16]:
results = retrieve("What is the TSA Gold Plus program?")

for result in results:
    print(f"\nScore: {result["score"]}")
    print(result["document"]["title"])


Score: 11.47537980628859
TSA's new 'Gold+' program looks to increase private security screening at airports

Score: 5.249376552637838
The Education Department is hiring — while it's being dismantled

Score: 4.046701465423335
Ebola fears surge on the ground in Congo over rapid spread of a rare type

Score: 3.8897554794219795
As floods get worse Britain tries a new solution: beavers

Score: 3.5692669284067113
Even as anxieties grow under Trump these swing voters aren't ready to back Democrats


# Convert to LangChain Docs

In [17]:
langchain_docs = []

for doc in documents:
    langchain_docs.append(
        Document(
            page_content= doc["text"],
            metadata={
                "title": doc["title"],
                "node_id": doc["node_id"],
                "level": doc["level"],
                "parent": doc["parent"]
            }
        )
    )

# Custum Retriever

In [ ]:
# Custom retriever class that implements LangChain's BaseRetriever interface
# This allows our BM25 index to work with LangChain's retrieval chains
class BM25Retriever(BaseRetriever):
    # Pydantic fields - required to be declared for LangChain's BaseRetriever
    bm25: BM25Okapi = Field(...)  # BM25 index instance
    documents: list = Field(...)   # List of documents to retrieve from
    k: int = 5                     # Number of top documents to return

    def _get_relevant_documents(self, query: str):
        # This method is required by BaseRetriever interface
        # It takes a query and returns the top-k matching documents
        query_tokens = (query.lower().split())
        scores = self.bm25.get_scores(query_tokens)
        # Get indices of top-k highest-scoring documents
        top_idx = np.argsort(scores)[::-1][:self.k]

        return [self.documents[i] for i in top_idx]

In [19]:
retriever = BM25Retriever(bm25=bm25, documents=langchain_docs, k=5)

# Gemini and Prompt

In [20]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

In [ ]:
# Define prompt template for RAG system
# This instructs the model to use ONLY the provided context for answers
# and prevents hallucination from the model's training data
prompt = ChatPromptTemplate.from_template(
"""
You are a retrieval-based assistant.

Answer only using the supplied context.

If the answer is unavailable
in the context, say:

"I could not find the answer in the provided documents."

Context:
{context}

Question:
{input}

Provide:
1. Answer
2. Supporting evidence
"""
)

# Document and Retreivel Chain

In [ ]:
# Build the RAG chain in two steps
# Step 1: Document chain - formats retrieved documents into the prompt
# Step 2: Retrieval chain - orchestrates retriever + document chain + LLM

# Document chain takes retrieved documents and passes them with the prompt to LLM
document_chain = (
    create_stuff_documents_chain(
        llm,
        prompt
    )
)

# Retrieval chain coordinates the entire RAG pipeline:
# Query -> Retriever -> Documents -> Document Chain -> LLM -> Answer
retrieval_chain = (
    create_retrieval_chain(
        retriever,
        document_chain
    )
)

# Test

In [23]:
question = ("Why is Britain using beavers to reduce flooding?")

response = retrieval_chain.invoke({"input": question})

print(response["answer"])

1. **Answer:** Britain is using beavers to reduce flooding because climate change is causing heavier and more erratic rainfall, leading to increased waterlogging in areas that previously did not flood. Scientists have enlisted beavers, known as "the animal kingdom's best flood engineers," to help mitigate this modern problem.

2. **Supporting evidence:**
    * "Britain is famous for drizzle but climate change is making rainfall heavier and more erratic places that didn't used to flood are now waterlogged so scientists have enlisted some of the animal kingdom s best flood engineers beavers to help in west london..."
    * "within weeks the beavers dammed up the creek creating a pond that holds water and stops it from spilling into the city they also diverted the creek s flow into smaller tributaries creating a wetland that better absorbs heavy rainfall mitigating the risk of flooding downstream they effectively turned this site into a giant sponge that can take heavy rainfall and slowly

In [24]:
full_corpus = "\n\n".join([doc.page_content for doc in langchain_docs])

In [ ]:
# Naive approach: Pass entire corpus as context without any retrieval
# This is the baseline for comparison - slower but includes all information
start = time.time()

naive_response = llm.invoke(
    f"""
Context:
{full_corpus}

Question:
{question}
"""
)

naive_time = (time.time() - start)  # Measure execution time

print(naive_response.content)

print(f"Naive Time: {naive_time}")

Britain is using beavers to reduce flooding because climate change is causing heavier and more erratic rainfall, leading to increased waterlogging in areas that didn't previously flood.

Beavers are considered "flood engineers" and help by:
*   **Building dams:** This creates ponds and diverts creek flows into smaller tributaries.
*   **Creating wetlands:** The ponds and wetlands act like "giant sponges" that can absorb heavy rainfall and slowly release water back into the landscape.
*   **Mitigating downstream flooding:** This process increases resilience against flooding and has even allowed cities to scrap expensive plans for reservoirs and levees, offering a more cost-effective and sustainable solution.
Naive Time: 9.803037166595459


In [ ]:
# RAG approach: Retrieve relevant documents first, then pass only top-k to LLM
# This should be faster because fewer tokens are passed to the model
start = time.time()

rag_response = retrieval_chain.invoke({"input": question})

rag_time = (time.time() - start)  # Measure execution time

print(rag_response["answer"])

print(f"RAG Time: {rag_time}")

1. **Answer:** Britain is using beavers to reduce flooding because climate change is causing heavier and more erratic rainfall, leading to places that didn't previously flood becoming waterlogged. Scientists have enlisted beavers, known as "animal kingdom's best flood engineers," to help mitigate this problem.

2. **Supporting evidence:**
    * "Britain is famous for drizzle but climate change is making rainfall heavier and more erratic places that didn't used to flood are now waterlogged so scientists have enlisted some of the animal kingdom s best flood engineers beavers to help"
    * "within weeks the beavers dammed up the creek creating a pond that holds water and stops it from spilling into the city they also diverted the creek s flow into smaller tributaries creating a wetland that better absorbs heavy rainfall mitigating the risk of flooding downstream they effectively turned this site into a giant sponge that can take heavy rainfall and slowly release water back into the lands

# Final Eval and Summary

In [27]:
questions = [
    "Why are Ebola fears increasing in Congo?",
    "Why are prediction markets becoming popular again?",
    "What concerns do swing voters have about Democrats?",
    "Why is the Education Department hiring while being dismantled?"
]

In [ ]:
# Run comprehensive evaluation on all test questions
# For each question, measure both RAG and naive approach latency and answers
results = []

for q in questions:
    # RAG approach: retrieval + inference
    rag_start = time.time()
    rag_answer = retrieval_chain.invoke({"input": q})["answer"]
    rag_latency = (time.time() - rag_start)

    # Naive approach: full context + inference
    naive_start = time.time()
    naive_answer = llm.invoke(
        f"""
Context:
{full_corpus}

Question:
{q}
"""
    ).content
    naive_latency = (time.time() - naive_start)

    # Store results for analysis
    results.append(
        {
            "question":     q,
            "rag_time":     rag_latency,
            "naive_time":   naive_latency,
            "rag_answer":   rag_answer,
            "naive_answer": naive_answer
        }
    )

In [29]:
avg_rag = np.mean([r["rag_time"] for r in results])

avg_naive = np.mean([r["naive_time"] for r in results])

print(f"Average RAG Time: {avg_rag}")

print(f"Average Naive Time: {avg_naive}")

Average RAG Time: 7.318785095214844
Average Naive Time: 10.727954006195068


In [30]:
with open("evaluation_results.json", "w", encoding="utf-8") as f:

    json.dump(
        results,
        f,
        indent=4,
        ensure_ascii=False
    )

print("Saved evaluation_results.json")

Saved evaluation_results.json
